# Vision Transformer (ViT) PASCAL VOC Classification Example

This notebook is an English, GitHub-friendly version of the original article. The code is kept unchanged, while the surrounding explanations have been translated and reorganized for easier reuse.

Original reference:
......................................................................................................................................................................

This is an educational example of using a pretrained Vision Transformer (ViT) for PASCAL VOC 2012 image classification. It demonstrates both the zero-shot-style behavior of the original ImageNet head and a simple fine-tuning workflow for adapting the model to Pascal VOC multi-label classification.

The pretrained ViT-Base model uses a simple CNN-based patch projection module to split a 224×224 image into 16×16 patches, producing a 14×14 grid of embeddings with 768 dimensions. These embeddings are then processed by the Transformer encoder for classification.

This notebook shows two scenarios:
1. Using the pretrained ViT model without fine-tuning.
2. Fine-tuning the model on the PASCAL VOC 2012 dataset.


## 1. Download the PASCAL VOC dataset and import the required libraries

The code below downloads the dataset and prepares a few sample images for later visualization.


In [ ]:
import os
import random
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.datasets import VOCDetection
from PIL import Image
from transformers import ViTImageProcessor, ViTForImageClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score
import shutil

# ------------------------------
# 1. Download & prepare Pascal VOC 2012
# ------------------------------

data_dir = './data'
os.makedirs(data_dir, exist_ok=True)

# Download the dataset (this may take a while)
voc_train = VOCDetection(
    root=data_dir,
    year='2012',
    image_set='trainval',      # use train+validation for a larger training set
    download=True,
    transform=None
)

# Save a few random images for later visualisation
sample_dir = './sample_images'
os.makedirs(sample_dir, exist_ok=True)
random_indices = random.sample(range(len(voc_train)), 2)

for i, idx in enumerate(random_indices):
    img, _ = voc_train[idx]
    img.save(os.path.join(sample_dir, f'sample_{i}.png'))

print(f"Dataset ready. Total training images: {len(voc_train)}")


## 2. Prepare the dataset for training

For image preprocessing, this notebook uses an **apply-on-the-fly** transformation strategy. Although this introduces a small runtime overhead, it is a common choice for large datasets, especially when training happens on the GPU while the CPU has spare capacity. It also reduces memory usage because images are transformed only when needed.


In [ ]:
# ------------------------------
# 2. Build a dataset for multi-label classification
# ------------------------------

# The 20 Pascal VOC classes (in the order used by the dataset)
VOC_CLASSES = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]

class VOCClassificationDataset(Dataset):
    def __init__(self, root, split='trainval', transform=None):
        self.dataset = VOCDetection(
            root=root,
            year='2012',
            image_set=split,
            download=False,
            transform=None
        )
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, annotation = self.dataset[idx]
        # Build a multi-hot label vector
        label_vector = torch.zeros(len(VOC_CLASSES), dtype=torch.float32)
        objects = annotation['annotation']['object']
        if isinstance(objects, dict):
            objects = [objects]
        for obj in objects:
            class_name = obj['name']
            if class_name in VOC_CLASSES:
                class_idx = VOC_CLASSES.index(class_name)
                label_vector[class_idx] = 1.0
        if self.transform:
            img = self.transform(img)
        return img, label_vector

# Define preprocessing for ViT (normalisation)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = VOCClassificationDataset(root=data_dir, split='trainval', transform=transform)
val_dataset = VOCClassificationDataset(root=data_dir, split='val', transform=transform)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")


## 3. Use the pretrained Vision Transformer before fine-tuning

The pretrained ViT-Base model was trained on ImageNet-1K. It includes the built-in `id2label` mapping from class index to text label (`model.config.id2label[idx_class]`), which can be used directly.

In the unmodified state, the model can already produce multi-label-style predictions. The labels may be more detailed than those from a traditional CNN, while still capturing the main objects in the image.

The code below first evaluates the original ImageNet classifier head, then replaces it with a 20-class Pascal VOC head in preparation for fine-tuning.


In [ ]:
# ------------------------------
# Helper: denormalisation (used for display only)
# ------------------------------
def denormalize(tensor, mean, std):
    """Reverse the normalisation to get pixel values in [0,1]."""
    tensor = tensor.clone()
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    return tensor.clamp_(0, 1)

# ------------------------------
# 3. Load ViT-Base with ORIGINAL ImageNet head, predict, THEN replace head
# ------------------------------

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load the pre-trained model WITHOUT changing the head (keeps 1000-class ImageNet head)
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')
# It contained the default id2label from the 1000 ImageNet classes
model.to(device)
model.eval()

# Define the image processor
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')

# Pick two random images from the training set
random_indices = random.sample(range(len(train_dataset)), 2)

# ----- PART A: Predict using the original ImageNet head -----
for i, idx in enumerate(random_indices):
    img_tensor, label_vector = train_dataset[idx]   # Already normalised (ImageNet stats)

    # Denormalise for correct visualisation
    img_denorm = denormalize(img_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    img_pil = transforms.ToPILImage()(img_denorm)

    # Prepare input for the model
    inputs = processor(images=img_pil, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits          # Shape: [1, 1000] (ImageNet logits)
        probs = torch.nn.functional.softmax(logits, dim=-1).cpu().numpy()[0]

    # Show the image
    plt.figure(figsize=(6,6))
    plt.imshow(img_pil)
    plt.title(f"Original ViT (ImageNet) prediction for sample {i+1}")
    plt.axis('off')
    plt.show()

    # Get top-5 ImageNet predictions
    top5_idx = np.argsort(probs)[-5:][::-1]
    top5_labels = [model.config.id2label[idx] for idx in top5_idx]
    top5_probs = [probs[idx] for idx in top5_idx]

    # Ground truth VOC classes
    true_voc_classes = [VOC_CLASSES[j] for j, val in enumerate(label_vector.numpy()) if val == 1]

    print(f"Sample {i+1}:")
    print(f"  True VOC classes: {true_voc_classes if true_voc_classes else 'none'}")
    print(f"  Top-5 ImageNet predictions:")
    for label, prob in zip(top5_labels, top5_probs):
        print(f"    {label}: {prob:.3f}")
    print("-" * 50)

# ----- PART B: Replace the classifier head with a randomly initialised 20-class head -----
print("Replacing the classifier head with a randomly initialised 20-class head for Pascal VOC...")
model.classifier = nn.Linear(model.config.hidden_size, len(VOC_CLASSES))
# The new head is now randomly initialised (PyTorch default)
model.to(device)

# Now the model is ready for fine-tuning on Pascal VOC (to be done in Block 4)


## 4. Fine-tune the Vision Transformer

This section adapts the model to the target task. In the original article, the training accuracy quickly exceeded many traditional CNN-based approaches and reached very high performance after fine-tuning.


In [ ]:
# ------------------------------
# 4. Fine-tuning the model
# ------------------------------

# DataLoaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# Optimiser and loss
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
criterion = nn.BCEWithLogitsLoss()

num_epochs = 3

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(pixel_values=images).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)

    epoch_train_loss = total_loss / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)

    # Validation
    model.eval()
    total_val_loss = 0.0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(pixel_values=images).logits
            loss = criterion(outputs, labels)
            total_val_loss += loss.item() * images.size(0)
            preds = torch.sigmoid(outputs).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    epoch_val_loss = total_val_loss / len(val_loader.dataset)
    val_losses.append(epoch_val_loss)

    # Compute threshold-based accuracy
    all_preds = np.array(all_preds) > 0.5
    all_labels = np.array(all_labels)
    accuracy = (all_preds == all_labels).mean()

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train loss: {epoch_train_loss:.4f} | Val loss: {epoch_val_loss:.4f}")
    print(f"  Val multi-label accuracy: {accuracy:.4f}\\n")


## 5. Save the fine-tuned model and test it again

The fine-tuned weights are saved, reloaded, and then evaluated on the same example images used earlier.


In [ ]:
# ------------------------------
# 5. Save, reload, and test the fine-tuned model
# ------------------------------

# Save the fine-tuned model
model_save_path = './vit_pascal_voc_finetuned.pth'
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to {model_save_path}")

# Re-load the model
model_reloaded = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')
model_reloaded.classifier = nn.Linear(model_reloaded.config.hidden_size, len(VOC_CLASSES))
model_reloaded.load_state_dict(torch.load(model_save_path, map_location=device))
model_reloaded.to(device)
model_reloaded.eval()

# Denormalisation function (already defined, but kept here for clarity)
def denormalize(tensor, mean, std):
    tensor = tensor.clone()
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    return tensor.clamp_(0, 1)

# Test on the same two images as before (random_indices from block 3)
for i, idx in enumerate(random_indices):
    img_tensor, label_vector = train_dataset[idx]   # Already normalized

    # Denormalize to [0,1] before converting to PIL
    img_denorm = denormalize(img_tensor, mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    img_pil = transforms.ToPILImage()(img_denorm)

    # Processor applies its own correct normalization
    inputs = processor(images=img_pil, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model_reloaded(**inputs)
        logits = outputs.logits
        predictions = torch.sigmoid(logits).cpu().numpy()[0]

    plt.figure(figsize=(6,6))
    plt.imshow(img_pil)
    plt.title(f"Fine-tuned prediction for sample {i+1}")
    plt.axis('off')
    plt.show()

    predicted_classes = [VOC_CLASSES[j] for j, prob in enumerate(predictions) if prob > 0.5]
    true_classes = [VOC_CLASSES[j] for j, val in enumerate(label_vector.numpy()) if val == 1]

    print(f"Sample {i+1} (after fine-tuning):")
    print(f"  True classes: {true_classes if true_classes else 'none'}")
    print(f"  Predicted classes: {predicted_classes if predicted_classes else 'none'}\\n")


---

Contact:
- For job opportunities or project collaboration: `yucongcai_business@outlook.com`
- For research-related matters: `yucongcai_research@outlook.com`


---

## Version log

| Version | Date | Change |
|---|---|---|
| v1.0 | 2026-08-03 | Initial rebuild from `assets/previous-resources/` (archive kept untouched). |

### v1.0 changes (2026-08-03)

| Change |
|---|
| No code changes needed (clean notebook) |